# MLP Classifier for Terrain Classification with Strict LORO-CV

Multi-Layer Perceptron neural network for terrain classification using PyTorch.
This notebook implements the author's first neural network experiment and follows the same strict leave-one-run-out cross-validation (LORO-CV) protocol established in `07_baselines.ipynb`.

**Key features:**
- PyTorch MLP with configurable architecture (BatchNorm, ReLU, Dropout)
- Early stopping on stratified validation split (never leaks to test run)
- Strict LORO-CV: one run held out per fold, all pre-processing inside each fold
- Fold-local mutual information feature selection
- Permutation-based feature importance on held-out test sets
- Comprehensive metrics: accuracy, macro-F1, per-class precision/recall/F1, confusion matrices

**Data splits:** 5-run LORO-CV (run 3 split into part A and part B at temporal midpoint)
**Features:** 5-class terrain (cobblestone, dry_dirt_track, grass, muddy_dirt_track, smooth_terrain)

In [1]:
from __future__ import annotations

import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset

# Configure plotting
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# ============================================================================
# ENVIRONMENT SUMMARY: Print PyTorch version, device, and Python version
# ============================================================================
print("=" * 80)
print("ENVIRONMENT SUMMARY - MLP Classifier for Terrain Classification")
print("=" * 80)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("=" * 80)

# Project bootstrap: ensure src/ is in path for imports
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================================
# CONSTANTS: Same schema as 07_baselines.ipynb
# ============================================================================
LABEL_COL = "label"
RUN_COL = "run_id"
ID_COLS = [
    "window_id",
    "run_id",
    "segment_id",
    "label",
    "t_start",
    "t_end",
    "n_samples",
]

# 5-class terrain labels (same order as baselines for consistency)
LABEL_ORDER = [
    "cobblestone",
    "dry_dirt_track",
    "grass",
    "muddy_dirt_track",
    "smooth_terrain",
]

# After the 5-run split (run 3 → partA + partB), no test fold has missing classes
# This map is intentionally empty; Cell 2 will enforce strict matching.
KNOWN_DEGENERATE = {}

# Speed regime confounds for cobblestone (noted but less severe after split)
KNOWN_CONFOUNDED = {
    "log_20260309_141435.414_partB": {
        "cobblestone": "train=run4 only, test=partB (same session, consistent speed)",
    },
    "log_20260326_120021.508": {
        "cobblestone": "train=partB only, test=run4 (different session)",
    },
}

# ============================================================================
# PATHS AND OUTPUTS
# ============================================================================
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_A_pruned_5run.csv"
TOP_MI_PATH = PROJECT_ROOT / "data" / "processed" / "top_mi_features.json"
REPORTS_DIR = PROJECT_ROOT / "reports" / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

for out_dir in [REPORTS_DIR, RESULTS_DIR]:
    out_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# LOAD DATASET AND TOP MI FEATURES
# ============================================================================
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {DATASET_PATH}")
if not TOP_MI_PATH.exists():
    raise FileNotFoundError(f"Missing MI feature list: {TOP_MI_PATH}")

df = pd.read_csv(DATASET_PATH)

with TOP_MI_PATH.open("r", encoding="utf-8") as f:
    top_mi_raw = json.load(f)

# Parse top_mi_features.json (flexible format: dict or list)
if isinstance(top_mi_raw, dict):
    if "top_features" in top_mi_raw and isinstance(top_mi_raw["top_features"], list):
        top_mi_features = list(top_mi_raw["top_features"])
    else:
        top_mi_features = list(top_mi_raw.keys())
elif isinstance(top_mi_raw, list):
    top_mi_features = list(top_mi_raw)
else:
    raise ValueError("Unsupported top_mi_features.json structure. Expected list or dict.")

# ============================================================================
# VALIDATION: Schema check, label check, run count check
# ============================================================================
missing_id_cols = [c for c in ID_COLS if c not in df.columns]
if missing_id_cols:
    raise KeyError(f"Missing ID columns in dataset: {missing_id_cols}")

if LABEL_COL not in df.columns or RUN_COL not in df.columns:
    raise KeyError(f"Dataset must contain {LABEL_COL} and {RUN_COL}.")

labels_found = sorted(df[LABEL_COL].astype(str).unique().tolist())
if set(labels_found) != set(LABEL_ORDER):
    raise ValueError(
        "Strict label check failed. Expected labels: "
        f"{sorted(LABEL_ORDER)} | Found: {labels_found}"
    )

run_ids = sorted(df[RUN_COL].astype(str).unique().tolist())
if len(run_ids) < 2:
    raise ValueError("LORO-CV requires at least two unique runs.")

# Extract feature columns (exclude ID columns to get only sensor features)
feature_cols_all = [c for c in df.columns if c not in ID_COLS]
feature_cols_mi = [c for c in top_mi_features if c in feature_cols_all]
if not feature_cols_mi:
    raise ValueError("No overlap between top_mi_features.json and dataset columns.")

print(f"\nDataset loaded: {df.shape[0]} windows × {df.shape[1]} columns")
print(f"Feature count (all pruned): {len(feature_cols_all)}")
print(f"Feature count (top MI): {len(feature_cols_mi)}")
print(f"Runs: {run_ids}")
print(f"Classes: {LABEL_ORDER}")
print(f"\nLabel distribution:")
print(df[LABEL_COL].value_counts().reindex(LABEL_ORDER).fillna(0).astype(int).to_string())
print(f"\nWindows per run:")
print(df[RUN_COL].value_counts().sort_index().to_string())


ENVIRONMENT SUMMARY - MLP Classifier for Terrain Classification
PyTorch version: 2.11.0
Device: cpu
CUDA available: False
Python version: 3.11.14 | packaged by conda-forge | (main, Jan 27 2026, 00:01:01) [Clang 19.1.7 ]
NumPy version: 2.4.2
Pandas version: 3.0.0

Dataset loaded: 2346 windows × 130 columns
Feature count (all pruned): 123
Feature count (top MI): 9
Runs: ['log_20260223_142511.490', 'log_20260226_102148.990', 'log_20260309_141435.414_partA', 'log_20260309_141435.414_partB', 'log_20260326_120021.508']
Classes: ['cobblestone', 'dry_dirt_track', 'grass', 'muddy_dirt_track', 'smooth_terrain']

Label distribution:
label
cobblestone         355
dry_dirt_track      652
grass               488
muddy_dirt_track    145
smooth_terrain      706

Windows per run:
run_id
log_20260223_142511.490          293
log_20260226_102148.990          145
log_20260309_141435.414_partA    664
log_20260309_141435.414_partB    664
log_20260326_120021.508          580


## Cell 1: Hyperparameter Configuration

All tuneable parameters are defined in one place for easy experiment tracking and reproducibility.

In [2]:
# ============================================================================
# ARCHITECTURE PARAMETERS
# ============================================================================
# MLP layer widths: input_dim → HIDDEN_DIMS[0] → ... → HIDDEN_DIMS[-1] → num_classes
HIDDEN_DIMS = [256, 128, 64]      # Three hidden layers: 256 → 128 → 64 neurons
DROPOUT_RATE = 0.3                 # Fraction of activations set to zero (regularization)
USE_BATCH_NORM = True              # Apply BatchNorm after each linear layer (before ReLU)

# ============================================================================
# TRAINING PARAMETERS
# ============================================================================
LEARNING_RATE = 1e-3               # Adam optimizer step size; tune if loss plateaus
BATCH_SIZE = 32                    # Mini-batch size per gradient update
MAX_EPOCHS = 200                   # Maximum training epochs (early stopping may stop earlier)
EARLY_STOPPING_PATIENCE = 20       # Stop if validation loss doesn't improve for N epochs
VAL_SPLIT = 0.15                  # Stratified validation split: 15% of training fold reserved for validation
WEIGHT_DECAY = 1e-4               # L2 regularization coefficient (aids generalization)
USE_LR_SCHEDULER = True            # Enable CosineAnnealingLR to decay LR over training
USE_CLASS_WEIGHTS = True           # Weight CrossEntropyLoss by inverse class frequency (helps imbalanced classes)

# ============================================================================
# FEATURE SELECTION PARAMETERS
# ============================================================================
N_TOP_MI = 25                      # Select top 25 features by mutual information (per fold)
ENABLE_FOLD_LOCAL_TOP_MI = True    # Compute top MI features fresh for each fold (prevent leakage)
MI_NEIGHBORS = 5                   # KNN neighbors for mutual_info_classif (MI estimator parameter)
ENABLE_FEATURE_ABLATIONS = True    # If True, also evaluate "all_no_speed" feature set

# ============================================================================
# EVALUATION PARAMETERS
# ============================================================================
N_PERMUTATION_REPEATS = 8          # Number of random shuffles for permutation importance
MIN_WORST_FOLD_MACRO_F1 = 0.60    # Robustness threshold: worst fold must reach this F1
CLASS_SAMPLE_WARN_THRESHOLD = 30   # Warn if test fold has fewer than this many samples per class
TRAIN_COUNT_LOW_THRESHOLD = 80     # Flag: training count between LOW and CRITICAL
TRAIN_COUNT_CRITICAL_THRESHOLD = 30 # Flag: training count below this is critical

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
RANDOM_STATE = 42                  # Random seed for numpy, sklearn, PyTorch, Python random
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)

# ============================================================================
# OUTPUT OPTIONS
# ============================================================================
SAVE_MODELS = False                # If True, save trained models to disk (uses disk space)

print("\n" + "=" * 80)
print("HYPERPARAMETER CONFIGURATION")
print("=" * 80)
print(f"Architecture: {len(HIDDEN_DIMS)}-layer MLP")
print(f"  Hidden dims: {HIDDEN_DIMS}")
print(f"  Dropout rate: {DROPOUT_RATE}")
print(f"  Batch norm: {USE_BATCH_NORM}")
print(f"\nTraining:")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Validation split: {VAL_SPLIT * 100:.1f}% stratified")
print(f"  Weight decay (L2): {WEIGHT_DECAY}")
print(f"  LR scheduler (CosineAnnealing): {USE_LR_SCHEDULER}")
print(f"  Class-weighted loss: {USE_CLASS_WEIGHTS}")
print(f"\nFeatures:")
print(f"  Top MI features: {N_TOP_MI}")
print(f"  Fold-local MI: {ENABLE_FOLD_LOCAL_TOP_MI}")
print(f"  Permutation importance repeats: {N_PERMUTATION_REPEATS}")
print(f"  Feature ablations enabled: {ENABLE_FEATURE_ABLATIONS}")
print("=" * 80 + "\n")



HYPERPARAMETER CONFIGURATION
Architecture: 3-layer MLP
  Hidden dims: [256, 128, 64]
  Dropout rate: 0.3
  Batch norm: True

Training:
  Learning rate: 0.001
  Batch size: 32
  Max epochs: 200
  Early stopping patience: 20
  Validation split: 15.0% stratified
  Weight decay (L2): 0.0001
  LR scheduler (CosineAnnealing): True
  Class-weighted loss: True

Features:
  Top MI features: 25
  Fold-local MI: True
  Permutation importance repeats: 8
  Feature ablations enabled: True



## Cell 2: PyTorch Model Architecture

Define the neural network module and dataset class for PyTorch training.

In [3]:
class TerrainMLP(nn.Module):
    """
    Multi-Layer Perceptron for terrain classification.
    
    Architecture: Linear → BatchNorm → ReLU → Dropout → ... → Linear (output logits, no softmax)
    
    CrossEntropyLoss (used in training) applies softmax internally, so we output raw logits.
    This avoids numerical instability from log(softmax(softmax(x))).
    
    Args:
        input_dim: Number of input features
        hidden_dims: List of hidden layer widths, e.g., [256, 128, 64]
        num_classes: Number of output classes
        dropout_rate: Fraction of activations to zero (regularization)
        use_batch_norm: If True, apply BatchNorm after each hidden layer
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        num_classes: int,
        dropout_rate: float = 0.3,
        use_batch_norm: bool = True,
    ):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate
        self.use_batch_norm = use_batch_norm
        
        layers = []
        prev_dim = input_dim
        
        # Build hidden layers: Linear → BatchNorm → ReLU → Dropout
        for hidden_dim in hidden_dims:
            # Linear: prev_dim → hidden_dim
            layers.append(nn.Linear(prev_dim, hidden_dim))
            
            # BatchNorm: stabilizes activations, enables higher learning rates
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            
            # ReLU: non-linearity (max(0, x))
            layers.append(nn.ReLU())
            
            # Dropout: regularization (randomly zero activations during training)
            layers.append(nn.Dropout(p=dropout_rate))
            
            prev_dim = hidden_dim
        
        # Build the sequential module from layers list
        self.hidden_layers = nn.Sequential(*layers)
        
        # Output layer: hidden_dims[-1] → num_classes
        # No activation here; CrossEntropyLoss expects raw logits
        self.output_layer = nn.Linear(prev_dim, num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, input_dim)
        
        Returns:
            Logits of shape (batch_size, num_classes) — raw class scores, not probabilities
        """
        # Pass through hidden layers (with BatchNorm, ReLU, Dropout)
        h = self.hidden_layers(x)
        
        # Output layer: logits (no softmax; CrossEntropyLoss applies it internally)
        logits = self.output_layer(h)
        
        return logits


class TerrainDataset(Dataset):
    """
    PyTorch Dataset wrapper for terrain sensor data.
    
    This class handles:
    - Converting numpy arrays to PyTorch tensors
    - FloatTensor for features (continuous values); LongTensor for labels (class indices)
    - Proper tensor shapes for batch processing
    
    Args:
        X: Feature matrix (n_samples, n_features) as numpy array
        y: Label vector (n_samples,) as numpy array
    """
    
    def __init__(self, X: np.ndarray, y: np.ndarray):
        """
        Initialize dataset.
        
        Args:
            X: Features (n_samples, n_features)
            y: Labels (n_samples,) — should be integer-encoded class indices
        """
        # Store features as float32 for PyTorch (neural networks use float precision)
        # Convert to tensor: shape (n_samples, n_features)
        self.X = torch.from_numpy(X.astype(np.float32))
        
        # Store labels as long (int64) for PyTorch's CrossEntropyLoss
        # CrossEntropyLoss expects LongTensor, not FloatTensor, for integer class indices
        # Shape (n_samples,)
        self.y = torch.from_numpy(y.astype(np.int64))
        
        # Quick validation
        assert self.X.shape[0] == self.y.shape[0], "X and y must have same length"
    
    def __len__(self) -> int:
        """Return total number of samples."""
        return len(self.X)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Get one sample.
        
        Args:
            idx: Sample index
        
        Returns:
            (X_sample, y_sample) where:
            - X_sample: FloatTensor of shape (n_features,) — feature vector
            - y_sample: LongTensor scalar — class index
        """
        return self.X[idx], self.y[idx]


print("\nModel classes defined: TerrainMLP, TerrainDataset")



Model classes defined: TerrainMLP, TerrainDataset


## Cell 3: Training Utilities and Helper Functions

Define functions for model training, validation, and learning loop management.

In [4]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """
    Train the model for one complete epoch.
    
    Args:
        model: PyTorch model instance
        loader: DataLoader for training batches
        optimizer: Optimizer (e.g., Adam) for parameter updates
        criterion: Loss function (e.g., CrossEntropyLoss)
        device: Device to run on ("cuda" or "cpu")
    
    Returns:
        Mean loss across all batches in the epoch
    """
    model.train()  # Set model to training mode (enables dropout, batch norm updates)
    total_loss = 0.0
    batch_count = 0
    
    for X_batch, y_batch in loader:
        # Move batch to device (GPU if available)
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Forward pass: compute logits
        logits = model(X_batch)
        
        # Compute loss: CrossEntropyLoss expects (batch_size, num_classes) logits
        #               and (batch_size,) target indices
        loss = criterion(logits, y_batch)
        
        # Backward pass: compute gradients
        optimizer.zero_grad()  # Clear previous gradients
        loss.backward()        # Backpropagation
        optimizer.step()       # Update parameters
        
        # Accumulate loss for averaging
        total_loss += loss.item()
        batch_count += 1
    
    mean_loss = total_loss / max(batch_count, 1)
    return mean_loss


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, np.ndarray]:
    """
    Evaluate the model on a validation or test set.
    
    Args:
        model: PyTorch model instance
        loader: DataLoader for evaluation batches
        criterion: Loss function
        device: Device to run on
    
    Returns:
        (mean_loss, predictions_array) where:
        - mean_loss: Average loss across batches
        - predictions_array: Predicted class indices (numpy, shape (n_samples,))
    """
    model.eval()  # Set to evaluation mode (disables dropout, batch norm uses running stats)
    total_loss = 0.0
    batch_count = 0
    all_predictions = []
    
    # torch.no_grad() disables gradient tracking (saves memory and speeds up forward pass)
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            # Forward pass
            logits = model(X_batch)
            
            # Compute loss
            loss = criterion(logits, y_batch)
            total_loss += loss.item()
            batch_count += 1
            
            # Get predicted class indices: argmax over class dimension (dim=1)
            # logits: (batch_size, num_classes) → predictions: (batch_size,)
            predictions = torch.argmax(logits, dim=1)
            all_predictions.append(predictions.cpu().numpy())
    
    mean_loss = total_loss / max(batch_count, 1)
    predictions_array = np.concatenate(all_predictions, axis=0)
    
    return mean_loss, predictions_array


def train_fold(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    config: dict,
    device: torch.device,
) -> tuple[nn.Module, dict]:
    """
    Train model on one LORO-CV fold with early stopping on validation loss.
    
    Args:
        model: Initialized TerrainMLP model
        X_train: Training features (n_train, n_features)
        y_train: Training labels (n_train,) — integer-encoded
        X_val: Validation features (n_val, n_features)
        y_val: Validation labels (n_val,)
        config: Dict with training hyperparameters:
            - learning_rate, batch_size, max_epochs, early_stopping_patience
        device: Device to run on
    
    Returns:
        (trained_model, history) where:
        - trained_model: Model with best validation weights
        - history: Dict with keys:
            - train_loss: list of mean losses per epoch
            - val_loss: list of validation losses per epoch
            - best_epoch: epoch with lowest validation loss
    """
    # Extract hyperparameters from config
    lr = config["learning_rate"]
    batch_size = config["batch_size"]
    max_epochs = config["max_epochs"]
    patience = config["early_stopping_patience"]
    weight_decay = config["weight_decay"]
    use_lr_scheduler = config.get("use_lr_scheduler", False)
    use_class_weights = config.get("use_class_weights", False)
    class_weights = config.get("class_weights", None)  # FloatTensor or None
    
    # Create DataLoaders: shuffle training, no shuffle for validation/test
    train_dataset = TerrainDataset(X_train, y_train)
    val_dataset = TerrainDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Setup optimizer: Adam with L2 regularization (weight_decay)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # LR scheduler: CosineAnnealingLR smoothly decays LR from lr → ~0 over max_epochs
    # This avoids the sharp step-drops of StepLR and often converges to a better minimum
    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=lr * 0.01)
        if use_lr_scheduler else None
    )
    
    # Loss function: optionally weight each class by inverse frequency to handle imbalance
    # class_weights[i] = N_total / (N_classes * N_i) — rarer classes get higher weight
    if use_class_weights and class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    else:
        criterion = nn.CrossEntropyLoss()
    
    # Training history
    history = {
        "train_loss": [],
        "val_loss": [],
        "lr_history": [],
        "best_epoch": -1,
    }
    
    best_val_loss = float("inf")
    patience_counter = 0
    best_state_dict = None
    
    # Main training loop
    for epoch in range(max_epochs):
        # Record current LR before the step
        current_lr = optimizer.param_groups[0]["lr"]
        history["lr_history"].append(current_lr)
        
        # Train one epoch
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        history["train_loss"].append(train_loss)
        
        # Evaluate on validation set
        val_loss, _ = evaluate(model, val_loader, criterion, device)
        history["val_loss"].append(val_loss)
        
        # Step LR scheduler after validation (so early stopping sees the final LR)
        if scheduler is not None:
            scheduler.step()
        
        # Early stopping: check if validation loss improved
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save model state when validation loss improves
            best_state_dict = model.state_dict().copy()
            history["best_epoch"] = epoch
        else:
            patience_counter += 1
        
        # Stop training if patience exceeded
        if patience_counter >= patience:
            break
    
    # Restore best model weights before returning
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    
    return model, history


def fold_local_top_mi_features(
    train_df: pd.DataFrame,
    y_train: np.ndarray,
    pool_cols: list[str],
    n_top: int,
    n_neighbors: int,
    random_state: int,
    fallback_cols: list[str],
) -> list[str]:
    """
    Select top mutual information features for this fold (same as 07_baselines.ipynb).
    
    Args:
        train_df: Training data DataFrame
        y_train: Training labels
        pool_cols: Candidate feature columns
        n_top: Number of top features to select
        n_neighbors: KNN neighbors for MI estimator
        random_state: Random seed
        fallback_cols: Fallback features if selection fails
    
    Returns:
        List of selected feature column names
    """
    if n_top <= 0:
        return list(fallback_cols)
    
    # Extract feature matrix, replace NaN and inf with 0
    X_pool = np.nan_to_num(
        train_df[pool_cols].to_numpy(dtype=float),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    
    if X_pool.shape[1] == 0:
        return list(fallback_cols)
    
    # Compute mutual information between each feature and target
    scores = mutual_info_classif(
        X_pool,
        y_train,
        n_neighbors=n_neighbors,
        random_state=random_state,
    )
    
    # Rank by MI score (descending) and select top N
    ranked_idx = np.argsort(scores)[::-1]
    selected = [pool_cols[i] for i in ranked_idx[: min(n_top, len(pool_cols))]]
    
    return selected if selected else list(fallback_cols)


def compute_fold_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    evaluable_classes: list[str],
    label_order: list[str],
) -> dict[str, object]:
    """
    Compute accuracy, macro F1, and per-class metrics (same as 07_baselines.ipynb).
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        evaluable_classes: Classes present in both train and test
        label_order: Canonical label order
    
    Returns:
        Dict with accuracy, macro_f1, and per_class metrics
    """
    y_true = np.asarray(y_true, dtype=object)
    y_pred = np.asarray(y_pred, dtype=object)
    
    accuracy = accuracy_score(y_true, y_pred)
    
    # Macro F1 only over evaluable classes (those present in both train and test)
    if evaluable_classes:
        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=evaluable_classes,
            average="macro",
            zero_division=0,
        )
    else:
        macro_f1 = np.nan
    
    # Per-class metrics
    p_all, r_all, f_all, s_all = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=label_order,
        zero_division=0,
    )
    
    per_class = {}
    evaluable_set = set(evaluable_classes)
    for idx, label in enumerate(label_order):
        if label in evaluable_set:
            per_class[label] = {
                "precision": float(p_all[idx]),
                "recall": float(r_all[idx]),
                "f1": float(f_all[idx]),
                "support": int(s_all[idx]),
            }
        else:
            per_class[label] = {
                "precision": np.nan,
                "recall": np.nan,
                "f1": np.nan,
                "support": int(s_all[idx]),
            }
    
    return {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1) if not np.isnan(macro_f1) else np.nan,
        "per_class": per_class,
        "n_test": int(len(y_true)),
    }


def row_normalize(cm: np.ndarray) -> np.ndarray:
    """Row-normalize confusion matrix (each row sums to 1)."""
    cm = cm.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums > 0)


# Wrapper class for sklearn's permutation_importance (expects sklearn-compatible predict method)
class TorchModelWrapper:
    """
    Wrap PyTorch model to make it compatible with sklearn's permutation_importance.
    
    sklearn's permutation_importance expects a fitted estimator with a predict() method
    that takes X and returns predictions. PyTorch models don't follow this interface by default.
    """
    
    def __init__(self, model: nn.Module, device: torch.device, label_encoder: LabelEncoder):
        """
        Args:
            model: Trained TerrainMLP model
            device: Device model is on
            label_encoder: LabelEncoder for converting encoded predictions back to class names
        """
        self.model = model
        self.device = device
        self.label_encoder = label_encoder
        self.model.eval()  # Set to evaluation mode
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict class labels for samples in X.
        
        Args:
            X: Feature array (n_samples, n_features)
        
        Returns:
            Predicted class labels as numpy array
        """
        X_tensor = torch.from_numpy(X.astype(np.float32)).to(self.device)
        
        with torch.no_grad():
            logits = self.model(X_tensor)
            predictions_encoded = torch.argmax(logits, dim=1).cpu().numpy()
        
        # Convert encoded predictions back to original class names
        predictions_decoded = self.label_encoder.inverse_transform(predictions_encoded)
        return predictions_decoded
    
    def fit(self, X, y):
        """
        No-op fit method (model is already trained) required by sklearn interface.
        """
        return self


print("Training utilities defined: train_one_epoch, evaluate, train_fold, etc.")


Training utilities defined: train_one_epoch, evaluate, train_fold, etc.


## Cell 4: LORO-CV Evaluation Loop

Iterate over feature sets and runs, train MLP for each fold, compute metrics, and store results.

In [5]:
# Prepare device: use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# ============================================================================
# SETUP FEATURE SETS
# ============================================================================
ODOM_FEATURE_COLS = [
    c for c in ["odom_mean_speed", "odom_speed_std", "odom_distance"]
    if c in feature_cols_all
]

feature_sets: dict[str, list[str]] = {
    "all": feature_cols_all,
    "top_mi": feature_cols_mi,
}

if ENABLE_FEATURE_ABLATIONS:
    all_no_speed = [c for c in feature_cols_all if c not in set(ODOM_FEATURE_COLS)]
    if all_no_speed:
        feature_sets["all_no_speed"] = all_no_speed

print(f"Feature sets to evaluate: {list(feature_sets.keys())}")
for fs_name, fs_cols in feature_sets.items():
    print(f"  {fs_name}: {len(fs_cols)} features")

# ============================================================================
# STORAGE FOR RESULTS AND ARTIFACTS
# ============================================================================
detail_rows: list[dict[str, object]] = []
per_class_rows: list[dict[str, object]] = []
confusion_records: list[dict[str, object]] = []
fold_label_encoders: dict[tuple[str, str], LabelEncoder] = {}
fold_top_mi_rows: list[dict[str, object]] = []
learning_curves_store: dict[str, list[dict]] = {}
permutation_importance_store: dict[str, list[pd.Series]] = {}

# ============================================================================
# MAIN EVALUATION LOOP: Feature Sets × Runs (LORO-CV)
# ============================================================================
print("\n" + "=" * 80)
print("STARTING LORO-CV EVALUATION")
print("=" * 80)

for feature_set_name, base_feature_cols in feature_sets.items():
    learning_curves_store[feature_set_name] = []
    
    for test_run in run_ids:
        # Split: training = all runs except test_run, test = test_run only
        train_mask = df[RUN_COL].astype(str) != test_run
        test_mask = ~train_mask
        
        y_train = df.loc[train_mask, LABEL_COL].astype(str).to_numpy()
        y_test = df.loc[test_mask, LABEL_COL].astype(str).to_numpy()
        
        # STRICT VALIDATION: KNOWN_DEGENERATE check
        expected_degenerate = sorted(KNOWN_DEGENERATE.get(test_run, []))
        observed_missing_from_train = sorted(set(y_test) - set(y_train))
        if expected_degenerate != observed_missing_from_train:
            raise ValueError(
                f"KNOWN_DEGENERATE mismatch during evaluation for {test_run}. "
                f"Expected {expected_degenerate}, observed {observed_missing_from_train}."
            )
        
        # ====================================================================
        # FEATURE SELECTION (per fold to prevent leakage)
        # ====================================================================
        selected_feature_cols = list(base_feature_cols)
        
        if feature_set_name == "top_mi" and ENABLE_FOLD_LOCAL_TOP_MI:
            # Compute fold-local MI on training data only
            train_df_full = df.loc[train_mask]
            selected_feature_cols = fold_local_top_mi_features(
                train_df=train_df_full,
                y_train=y_train,
                pool_cols=feature_cols_all,
                n_top=N_TOP_MI,
                n_neighbors=MI_NEIGHBORS,
                random_state=RANDOM_STATE,
                fallback_cols=feature_cols_mi,
            )
            fold_top_mi_rows.append(
                {
                    "test_run": test_run,
                    "n_features": int(len(selected_feature_cols)),
                    "selected_features": "|".join(selected_feature_cols),
                }
            )
        
        # ====================================================================
        # DATA PREPARATION (with StandardScaler fitted on training only)
        # ====================================================================
        X_train_raw = df.loc[train_mask, selected_feature_cols]
        X_test_raw = df.loc[test_mask, selected_feature_cols]
        
        # Fit StandardScaler on training data ONLY (prevent leakage)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)  # fit AND transform
        X_test_scaled = scaler.transform(X_test_raw)  # transform only (fitted on train)
        
        # ====================================================================
        # STRATIFIED VALIDATION SPLIT (within training fold)
        # ====================================================================
        # Split training data into train (85%) and validation (15%), stratified by label
        # This ensures class distribution is preserved in both subsets
        X_train, X_val, y_train_split, y_val = train_test_split(
            X_train_scaled,
            y_train,
            test_size=VAL_SPLIT,
            stratify=y_train,
            random_state=RANDOM_STATE,
        )
        
        # ====================================================================
        # LABEL ENCODING (fit on training labels only)
        # ====================================================================
        le = LabelEncoder()
        y_train_encoded = le.fit_transform(y_train_split)
        y_val_encoded = le.transform(y_val)
        y_test_encoded = le.transform(y_test)
        
        fold_label_encoders[(feature_set_name, test_run)] = le
        
        # ====================================================================
        # CLASS WEIGHTS: inverse-frequency weighting for CrossEntropyLoss
        # ====================================================================
        # class_weight[i] = N_total / (N_classes * count[i])
        # Rarer classes receive higher weight, which penalises misclassifying them more.
        num_classes_local = len(le.classes_)
        class_counts = np.bincount(y_train_encoded, minlength=num_classes_local).astype(float)
        class_weight_values = len(y_train_encoded) / (num_classes_local * np.maximum(class_counts, 1))
        class_weight_tensor = torch.tensor(class_weight_values, dtype=torch.float32)
        
        # ====================================================================
        # MODEL INITIALIZATION AND TRAINING
        # ====================================================================
        num_classes = len(le.classes_)
        num_features = X_train.shape[1]
        
        model = TerrainMLP(
            input_dim=num_features,
            hidden_dims=HIDDEN_DIMS,
            num_classes=num_classes,
            dropout_rate=DROPOUT_RATE,
            use_batch_norm=USE_BATCH_NORM,
        )
        model.to(device)
        
        # Train fold
        config = {
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "weight_decay": WEIGHT_DECAY,
            "use_lr_scheduler": USE_LR_SCHEDULER,
            "use_class_weights": USE_CLASS_WEIGHTS,
            "class_weights": class_weight_tensor,
        }
        
        trained_model, history = train_fold(
            model=model,
            X_train=X_train,
            y_train=y_train_encoded,
            X_val=X_val,
            y_val=y_val_encoded,
            config=config,
            device=device,
        )
        
        # ====================================================================
        # INFERENCE ON TEST SET
        # ====================================================================
        criterion = nn.CrossEntropyLoss()
        test_dataset = TerrainDataset(X_test_scaled, y_test_encoded)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        _, y_pred_encoded = evaluate(trained_model, test_loader, criterion, device)
        y_pred = le.inverse_transform(y_pred_encoded)
        
        # ====================================================================
        # COMPUTE FOLD METRICS
        # ====================================================================
        evaluable_classes = sorted(set(y_train).intersection(set(y_test)))
        fold_metrics = compute_fold_metrics(
            y_true=y_test,
            y_pred=y_pred,
            evaluable_classes=evaluable_classes,
            label_order=LABEL_ORDER,
        )
        
        best_epoch = history["best_epoch"]
        stopped_early = (best_epoch + 1) < MAX_EPOCHS
        
        # Print fold progress
        print(
            f"Fold {test_run} | feature_set={feature_set_name} | n_feat={len(selected_feature_cols)} "
            f"| acc={fold_metrics['accuracy']:.3f} | macro_f1={fold_metrics['macro_f1']:.3f} "
            f"| best_epoch={best_epoch} | stopped_early={stopped_early}"
        )
        
        # ====================================================================
        # STORE RESULTS
        # ====================================================================
        detail_rows.append(
            {
                "feature_set": feature_set_name,
                "test_run": test_run,
                "n_test": fold_metrics["n_test"],
                "n_features": int(len(selected_feature_cols)),
                "n_train": int(len(y_train_split)),
                "n_val": int(len(y_val)),
                "accuracy": fold_metrics["accuracy"],
                "macro_f1": fold_metrics["macro_f1"],
                "best_epoch": best_epoch,
                "early_stopped": stopped_early,
            }
        )
        
        # Store learning curves
        learning_curves_store[feature_set_name].append(
            {
                "test_run": test_run,
                "train_loss": history["train_loss"],
                "val_loss": history["val_loss"],
                "best_epoch": best_epoch,
            }
        )
        
        # Store confusion matrix data
        confusion_records.append(
            {
                "feature_set": feature_set_name,
                "test_run": test_run,
                "y_true": y_test.copy(),
                "y_pred": y_pred.copy(),
            }
        )
        
        # Store per-class metrics
        for cls_name in LABEL_ORDER:
            cls_metrics = fold_metrics["per_class"][cls_name]
            per_class_rows.append(
                {
                    "feature_set": feature_set_name,
                    "test_run": test_run,
                    "class": cls_name,
                    "precision": cls_metrics["precision"],
                    "recall": cls_metrics["recall"],
                    "f1": cls_metrics["f1"],
                    "support": cls_metrics["support"],
                }
            )
        
        # ====================================================================
        # PERMUTATION IMPORTANCE (on held-out test set)
        # ====================================================================
        try:
            wrapper = TorchModelWrapper(trained_model, device, le)
            perm_result = permutation_importance(
                estimator=wrapper,
                X=X_test_scaled,
                y=y_test,  # Use original class labels (not encoded)
                scoring="f1_macro",
                n_repeats=N_PERMUTATION_REPEATS,
                random_state=RANDOM_STATE,
                n_jobs=1,  # No multiprocessing with PyTorch models
            )
            perm_series = pd.Series(
                perm_result.importances_mean,
                index=selected_feature_cols,
                dtype=float,
            )
            perm_key = feature_set_name
            permutation_importance_store.setdefault(perm_key, []).append(perm_series)
        except Exception as exc:
            warnings.warn(f"Permutation importance failed for {test_run} ({feature_set_name}): {exc}")

print("\nLORO-CV evaluation completed.\n")

# Save fold-local MI selections if any
if fold_top_mi_rows:
    fold_top_mi_df = pd.DataFrame(fold_top_mi_rows)
    fold_top_mi_out = RESULTS_DIR / "mlp_fold_local_top_mi_features.csv"
    fold_top_mi_df.to_csv(fold_top_mi_out, index=False)
    print(f"Saved fold-local MI selections: {fold_top_mi_out}\n")


Using device: cpu

Feature sets to evaluate: ['all', 'top_mi', 'all_no_speed']
  all: 123 features
  top_mi: 9 features
  all_no_speed: 120 features

STARTING LORO-CV EVALUATION
Fold log_20260223_142511.490 | feature_set=all | n_feat=123 | acc=0.679 | macro_f1=0.597 | best_epoch=44 | stopped_early=True


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/963315541.py:277: UserWarning: Permutation importance failed for log_20260223_142511.490 (all): The following error was raised: 'TorchModelWrapper' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.
  warnings.warn(f"Permutation importance failed for {test_run} ({feature_set_name}): {exc}")


Fold log_20260226_102148.990 | feature_set=all | n_feat=123 | acc=0.883 | macro_f1=0.877 | best_epoch=31 | stopped_early=True


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/963315541.py:277: UserWarning: Permutation importance failed for log_20260226_102148.990 (all): The following error was raised: 'TorchModelWrapper' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.
  warnings.warn(f"Permutation importance failed for {test_run} ({feature_set_name}): {exc}")


Fold log_20260309_141435.414_partA | feature_set=all | n_feat=123 | acc=0.830 | macro_f1=0.862 | best_epoch=28 | stopped_early=True


...

/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/963315541.py:277: UserWarning: Permutation importance failed for log_20260309_141435.414_partA (all_no_speed): The following error was raised: 'TorchModelWrapper' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.
  warnings.warn(f"Permutation importance failed for {test_run} ({feature_set_name}): {exc}")


Fold log_20260309_141435.414_partB | feature_set=all_no_speed | n_feat=120 | acc=0.864 | macro_f1=0.784 | best_epoch=8 | stopped_early=True


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/963315541.py:277: UserWarning: Permutation importance failed for log_20260309_141435.414_partB (all_no_speed): The following error was raised: 'TorchModelWrapper' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.
  warnings.warn(f"Permutation importance failed for {test_run} ({feature_set_name}): {exc}")


Fold log_20260326_120021.508 | feature_set=all_no_speed | n_feat=120 | acc=0.455 | macro_f1=0.654 | best_epoch=40 | stopped_early=True

LORO-CV evaluation completed.

Saved fold-local MI selections: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/results/mlp_fold_local_top_mi_features.csv



/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/963315541.py:277: UserWarning: Permutation importance failed for log_20260326_120021.508 (all_no_speed): The following error was raised: 'TorchModelWrapper' object has no attribute '__sklearn_tags__'. It seems that there are no classes that implement `__sklearn_tags__` in the MRO and/or all classes in the MRO call `super().__sklearn_tags__()`. Make sure to inherit from `BaseEstimator` which implements `__sklearn_tags__` (or alternatively define `__sklearn_tags__` but we don't recommend this approach). Note that `BaseEstimator` needs to be on the right side of other Mixins in the inheritance order.
  warnings.warn(f"Permutation importance failed for {test_run} ({feature_set_name}): {exc}")


## Cell 5: Learning Curves and Results Tables

Visualize training dynamics and generate summary metrics tables.

In [6]:
# ============================================================================
# LEARNING CURVES: Visualize train vs validation loss per fold
# ============================================================================
# What to observe:
# - Gap between train and val loss → overfitting (model memorizes training data)
# - Both curves flat → underfitting or learning rate too low
# - Curves converging → good fit
# - Early stopping marked with dashed line → where training was halted

for feature_set_name, curves in learning_curves_store.items():
    if not curves:
        continue
    
    n_folds = len(curves)
    fig, axes = plt.subplots(
        (n_folds + 2) // 3, 3, figsize=(15, 4 * ((n_folds + 2) // 3)), 
        constrained_layout=True
    )
    
    if n_folds == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, curve_data in enumerate(curves):
        ax = axes[idx]
        test_run = curve_data["test_run"]
        train_loss = curve_data["train_loss"]
        val_loss = curve_data["val_loss"]
        best_epoch = curve_data["best_epoch"]
        
        # Plot loss curves
        epochs = range(len(train_loss))
        ax.plot(epochs, train_loss, label="Train loss", linewidth=1.5, alpha=0.8)
        ax.plot(epochs, val_loss, label="Val loss", linewidth=1.5, alpha=0.8)
        
        # Mark best epoch with vertical dashed line
        ax.axvline(best_epoch, color="red", linestyle="--", linewidth=1, alpha=0.7, label=f"Best epoch ({best_epoch})")
        
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(f"{test_run}")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(n_folds, len(axes)):
        axes[idx].axis("off")
    
    fig.suptitle(f"Learning Curves - {feature_set_name}", fontsize=14, y=1.00)
    lc_path = REPORTS_DIR / f"mlp_learning_curves_{feature_set_name}.png"
    fig.savefig(lc_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {lc_path}")

print()

# ============================================================================
# RESULTS SUMMARY TABLES
# ============================================================================
detail_df = pd.DataFrame(detail_rows)
if detail_df.empty:
    raise RuntimeError("No fold results were produced.")

print("Per-fold detail table:")
print(detail_df.to_string(index=False))

# Summarize each feature set
def summarize_group(group: pd.DataFrame) -> pd.Series:
    """Compute aggregated metrics across folds."""
    acc = group["accuracy"].dropna()
    mf1 = group["macro_f1"].dropna()
    
    acc_q75 = float(acc.quantile(0.75)) if not acc.empty else np.nan
    acc_q25 = float(acc.quantile(0.25)) if not acc.empty else np.nan
    mf1_q75 = float(mf1.quantile(0.75)) if not mf1.empty else np.nan
    mf1_q25 = float(mf1.quantile(0.25)) if not mf1.empty else np.nan
    
    macro_f1_mean = float(mf1.mean()) if not mf1.empty else np.nan
    macro_f1_std = float(mf1.std(ddof=0)) if not mf1.empty else np.nan
    
    # Robustness score: penalize high std deviation
    robustness_score = (
        macro_f1_mean - 0.5 * macro_f1_std
        if not np.isnan(macro_f1_mean) and not np.isnan(macro_f1_std)
        else np.nan
    )
    
    return pd.Series(
        {
            "accuracy_mean": float(acc.mean()) if not acc.empty else np.nan,
            "accuracy_std": float(acc.std(ddof=0)) if not acc.empty else np.nan,
            "accuracy_median": float(acc.median()) if not acc.empty else np.nan,
            "accuracy_iqr": acc_q75 - acc_q25 if not acc.empty else np.nan,
            "worst_fold_accuracy": float(acc.min()) if not acc.empty else np.nan,
            "macro_f1_mean": macro_f1_mean,
            "macro_f1_std": macro_f1_std,
            "macro_f1_median": float(mf1.median()) if not mf1.empty else np.nan,
            "macro_f1_iqr": mf1_q75 - mf1_q25 if not mf1.empty else np.nan,
            "worst_fold_macro_f1": float(mf1.min()) if not mf1.empty else np.nan,
            "robustness_score": robustness_score,
            "evaluable_folds": int(mf1.shape[0]),
        }
    )

summary_df = (
    detail_df.groupby(["feature_set"], as_index=False)
    .apply(summarize_group)
    .reset_index(drop=True)
)

summary_ranked = summary_df.sort_values("robustness_score", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("AGGREGATED SUMMARY TABLE")
print("=" * 80)
print(summary_df.to_string(index=False))

print("\nRanked by robustness_score (macro_f1_mean - 0.5 * macro_f1_std):")
print(
    summary_ranked[[
        "feature_set",
        "macro_f1_mean",
        "macro_f1_std",
        "worst_fold_macro_f1",
        "robustness_score",
    ]].to_string(index=False)
)

# Save results
detail_out = RESULTS_DIR / "mlp_metrics_per_fold.csv"
summary_out = RESULTS_DIR / "mlp_metrics_summary.csv"
ranking_out = RESULTS_DIR / "mlp_metrics_ranked_by_robustness.csv"

detail_df.to_csv(detail_out, index=False)
summary_df.to_csv(summary_out, index=False)
summary_ranked.to_csv(ranking_out, index=False)

print(f"\nSaved: {detail_out}")
print(f"Saved: {summary_out}")
print(f"Saved: {ranking_out}")


Saved: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/reports/models/mlp_learning_curves_all.png
Saved: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/reports/models/mlp_learning_curves_top_mi.png
Saved: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/reports/models/mlp_learning_curves_all_no_speed.png

Per-fold detail table:
 feature_set                      test_run  n_test  n_features  n_train  n_val  accuracy  macro_f1  best_epoch  early_stopped
         all       log_20260223_142511.490     293         123     1745    308  0.679181  0.597022          44           True
         all       log_20260226_102148.990     145         123     1870    331  0.882759  0.877352          31           True
         all log_20260309_141435.414_partA     664         123     1429    253  0.829819  0.861752          28           True
         all log_20260309_141435.414_partB     664         123     

## Cell 6: Confusion Matrices

Visualize per-fold and aggregated classification confusion matrices (row-normalized).

In [7]:
# For best feature set, plot per-fold and aggregated confusion matrices
best_feature_set = summary_ranked.iloc[0]["feature_set"]
print(f"Generating confusion matrices for best feature set: {best_feature_set}\n")

# Get all confusion records for this feature set
best_records = [r for r in confusion_records if r["feature_set"] == best_feature_set]

# Per-fold grid
fig, axes = plt.subplots(1, len(run_ids), figsize=(5 * len(run_ids), 4), constrained_layout=True)
if len(run_ids) == 1:
    axes = [axes]

for idx, test_run in enumerate(run_ids):
    ax = axes[idx]
    
    rec = next((r for r in best_records if r["test_run"] == test_run), None)
    if rec is None:
        ax.axis("off")
        ax.set_title(f"{test_run} (missing)")
        continue
    
    cm_raw = confusion_matrix(rec["y_true"], rec["y_pred"], labels=LABEL_ORDER)
    cm_norm = row_normalize(cm_raw)
    
    sns.heatmap(
        cm_norm,
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=LABEL_ORDER,
        yticklabels=LABEL_ORDER,
        cbar=False,
        annot=False,
    )
    
    # Mark confounded classes in this fold
    confounded_for_fold = KNOWN_CONFOUNDED.get(test_run, {})
    for conf_cls in confounded_for_fold.keys():
        if conf_cls in LABEL_ORDER:
            c_idx = LABEL_ORDER.index(conf_cls)
            ax.text(
                len(LABEL_ORDER) - 0.05,
                c_idx + 0.5,
                "⚡",
                ha="right",
                va="center",
                fontsize=10,
                color="darkorange",
            )
    
    ax.set_title(f"{test_run}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig.suptitle(f"MLP Confusion Matrices ({best_feature_set}) - Per Fold", fontsize=12)
per_fold_path = REPORTS_DIR / f"mlp_confusion_matrix_{best_feature_set}_per_fold.png"
fig.savefig(per_fold_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {per_fold_path}")

# Aggregated confusion matrix
agg_raw = np.zeros((len(LABEL_ORDER), len(LABEL_ORDER)), dtype=int)

for rec in best_records:
    cm_raw = confusion_matrix(rec["y_true"], rec["y_pred"], labels=LABEL_ORDER)
    agg_raw += cm_raw

agg_norm = row_normalize(agg_raw)

fig_agg, ax_agg = plt.subplots(figsize=(8, 6), constrained_layout=True)
sns.heatmap(
    agg_norm,
    ax=ax_agg,
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=LABEL_ORDER,
    yticklabels=LABEL_ORDER,
    cbar=True,
)
ax_agg.set_title(f"MLP ({best_feature_set}) - Aggregated across {len(run_ids)} folds")
ax_agg.set_xlabel("Predicted")
ax_agg.set_ylabel("True")

# Mark confounded classes
all_confounded_classes = set()
for fold_confounds in KNOWN_CONFOUNDED.values():
    all_confounded_classes.update(fold_confounds.keys())

for row_idx, cls_name in enumerate(LABEL_ORDER):
    if cls_name in all_confounded_classes:
        ax_agg.text(
            -0.05,
            row_idx + 0.5,
            "⚡",
            ha="right",
            va="center",
            fontsize=10,
            color="darkorange",
        )

ax_agg.text(
    0, -0.12,
    "⚡ = speed-regime confounded (cobblestone)",
    transform=ax_agg.transAxes,
    fontsize=8,
    color="gray",
)

agg_path = REPORTS_DIR / f"mlp_confusion_matrix_{best_feature_set}_aggregated.png"
fig_agg.savefig(agg_path, dpi=150, bbox_inches="tight")
plt.close(fig_agg)
print(f"Saved: {agg_path}")


Generating confusion matrices for best feature set: top_mi



/Users/pratyush/miniforge3/envs/ricbot/lib/python3.11/site-packages/seaborn/utils.py:61: UserWarning: Glyph 9889 (\N{HIGH VOLTAGE SIGN}) missing from font(s) Arial.
  fig.canvas.draw()
/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/3457786298.py:58: UserWarning: Glyph 9889 (\N{HIGH VOLTAGE SIGN}) missing from font(s) Arial.
  fig.savefig(per_fold_path, dpi=150, bbox_inches="tight")


Saved: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/reports/models/mlp_confusion_matrix_top_mi_per_fold.png
Saved: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/reports/models/mlp_confusion_matrix_top_mi_aggregated.png


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/3457786298.py:112: UserWarning: Glyph 9889 (\N{HIGH VOLTAGE SIGN}) missing from font(s) Arial.
  fig_agg.savefig(agg_path, dpi=150, bbox_inches="tight")


## Cell 7: Per-Class Metrics

Breakdown of precision, recall, and F1-score by terrain class.

In [8]:
per_class_df = pd.DataFrame(per_class_rows)

# Determine which classes are confounded across any fold
all_confounded_classes = set()
for fold_confounds in KNOWN_CONFOUNDED.values():
    all_confounded_classes.update(fold_confounds.keys())

# For best feature set
sub = per_class_df[per_class_df["feature_set"] == best_feature_set].copy()

rows = []
for cls_name in LABEL_ORDER:
    cls_sub = sub[sub["class"] == cls_name]
    f1_vals = cls_sub["f1"].dropna()
    folds_eval = int(f1_vals.shape[0])
    
    # Format mean ± std
    def fmt_mean_std(series):
        vals = series.dropna()
        if vals.empty:
            return "NaN"
        return f"{vals.mean():.3f} ± {vals.std(ddof=0):.3f}"
    
    warn = "WARN" if folds_eval < len(run_ids) else ""
    confound = "⚡ speed" if cls_name in all_confounded_classes else ""
    
    rows.append(
        {
            "Class": cls_name,
            "Precision": fmt_mean_std(cls_sub["precision"]),
            "Recall": fmt_mean_std(cls_sub["recall"]),
            "F1": fmt_mean_std(cls_sub["f1"]),
            "F1 median": float(f1_vals.median()) if not f1_vals.empty else np.nan,
            "F1 min": float(f1_vals.min()) if not f1_vals.empty else np.nan,
            "F1 max": float(f1_vals.max()) if not f1_vals.empty else np.nan,
            "Folds": f"{folds_eval}/{len(run_ids)}",
            "Flag": warn,
            "Confound": confound,
            "precision_mean": cls_sub["precision"].mean(skipna=True),
            "precision_std": cls_sub["precision"].std(ddof=0, skipna=True),
            "recall_mean": cls_sub["recall"].mean(skipna=True),
            "recall_std": cls_sub["recall"].std(ddof=0, skipna=True),
            "f1_mean": cls_sub["f1"].mean(skipna=True),
            "f1_std": cls_sub["f1"].std(ddof=0, skipna=True),
            "f1_median": float(f1_vals.median()) if not f1_vals.empty else np.nan,
            "f1_min": float(f1_vals.min()) if not f1_vals.empty else np.nan,
            "f1_max": float(f1_vals.max()) if not f1_vals.empty else np.nan,
            "folds_evaluated": folds_eval,
        }
    )

pretty_table = pd.DataFrame(rows)[[
    "Class",
    "Precision",
    "Recall",
    "F1",
    "F1 median",
    "F1 min",
    "F1 max",
    "Folds",
    "Flag",
    "Confound",
]]

print(f"\nPer-class metrics for MLP ({best_feature_set}):")
print(pretty_table.to_string(index=False))
print("\nNOTE: Classes with fewer than 5 evaluable folds are exploratory only.")
print("NOTE: ⚡ = cobblestone speed confound. Run 3 (partB) is high-speed; run 4 is low-speed.")

# Export to CSV
export_df = pd.DataFrame(rows)[[
    "Class",
    "precision_mean",
    "precision_std",
    "recall_mean",
    "recall_std",
    "f1_mean",
    "f1_std",
    "f1_median",
    "f1_min",
    "f1_max",
    "folds_evaluated",
    "Flag",
    "Confound",
]]
export_path = RESULTS_DIR / "mlp_per_class_metrics.csv"
export_df.to_csv(export_path, index=False)
print(f"\nSaved: {export_path}")



Per-class metrics for MLP (top_mi):
           Class     Precision        Recall            F1  F1 median   F1 min   F1 max Folds Flag Confound
     cobblestone 0.718 ± 0.282 0.387 ± 0.343 0.315 ± 0.231   0.314896 0.084337 0.545455   2/5 WARN  ⚡ speed
  dry_dirt_track 0.955 ± 0.022 0.839 ± 0.021 0.893 ± 0.022   0.893440 0.871901 0.914980   2/5 WARN         
           grass 0.789 ± 0.196 0.949 ± 0.025 0.847 ± 0.127   0.867133 0.632911 0.981891   5/5              
muddy_dirt_track 0.914 ± 0.014 0.328 ± 0.237 0.434 ± 0.269   0.433920 0.165138 0.702703   2/5 WARN         
  smooth_terrain 0.953 ± 0.050 0.968 ± 0.037 0.959 ± 0.037   0.960199 0.915966 1.000000   5/5              

NOTE: Classes with fewer than 5 evaluable folds are exploratory only.
NOTE: ⚡ = cobblestone speed confound. Run 3 (partB) is high-speed; run 4 is low-speed.

Saved: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/results/mlp_per_class_metrics.csv


## Cell 8: Permutation-Based Feature Importance

Analyze which features most impact model predictions via permutation importance on held-out test sets.

In [9]:
# ============================================================================
# PERMUTATION IMPORTANCE: Top features for best feature set
# ============================================================================
# Key insight: Permutation importance measures DROP in model performance when each 
# feature is randomly shuffled. For neural networks, this reflects how sensitive the 
# learned decision boundary is to each feature's values — NOT the magnitude of learned weights.

perm_key = best_feature_set
fold_perm_importances = permutation_importance_store.get(perm_key, [])

if fold_perm_importances:
    # Concatenate all fold importances and compute mean
    perm_df = pd.concat(fold_perm_importances, axis=1).T.fillna(0.0)
    perm_mean = perm_df.mean(axis=0).sort_values(ascending=False)
    perm_std = perm_df.std(axis=0, ddof=0)
    
    # Top 20 features
    top20_perm = perm_mean.head(20).sort_values(ascending=True)
    top20_std = perm_std[top20_perm.index]
    
    fig_perm, ax_perm = plt.subplots(figsize=(9, 7), constrained_layout=True)
    ax_perm.barh(
        range(len(top20_perm)),
        top20_perm.values,
        xerr=top20_std.values,
        alpha=0.7,
    )
    ax_perm.set_yticks(range(len(top20_perm)))
    ax_perm.set_yticklabels(top20_perm.index)
    ax_perm.set_xlabel("Mean Permutation Importance\n(F1-macro drop when feature shuffled)")
    ax_perm.set_ylabel("Feature")
    ax_perm.set_title(f"Top 20 Features - {best_feature_set}\n(Aggregated across {len(fold_perm_importances)} folds)")
    ax_perm.grid(True, alpha=0.3, axis="x")
    
    out_perm = REPORTS_DIR / "mlp_feature_importance_permutation.png"
    fig_perm.savefig(out_perm, dpi=150, bbox_inches="tight")
    plt.close(fig_perm)
    print(f"Saved: {out_perm}")
    
    print(f"\nTop 10 permutation-important features ({best_feature_set}):")
    for i, (feat, import_val) in enumerate(perm_mean.head(10).items(), 1):
        print(f"  {i}. {feat}: {import_val:.4f}")
else:
    warnings.warn(f"No stored permutation importances for {perm_key}.")


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_8024/2446299835.py:44: UserWarning: No stored permutation importances for top_mi.
  warnings.warn(f"No stored permutation importances for {perm_key}.")


## Cell 9: Baseline Comparison and Thesis Summary

Compare MLP results against 07_baselines.ipynb (LogisticRegression, RandomForest, XGBoost) and Dataset B (if available).

In [10]:
# ============================================================================
# LOAD AND COMPARE WITH BASELINE MODELS
# ============================================================================
baseline_path = RESULTS_DIR / "baseline_metrics_summary.csv"
if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    
    # Merge baseline and MLP results on feature_set
    comparison_data = []
    for feature_set_name in feature_sets.keys():
        baseline_row = baseline_df[baseline_df["feature_set"] == feature_set_name]
        mlp_row = summary_df[summary_df["feature_set"] == feature_set_name]
        
        if baseline_row.empty or mlp_row.empty:
            continue
        
        # Get best baseline model for this feature set
        baseline_best = baseline_row.sort_values("macro_f1_mean", ascending=False).iloc[0]
        mlp = mlp_row.iloc[0]
        
        delta_f1 = mlp["macro_f1_mean"] - baseline_best["macro_f1_mean"]
        delta_robust = mlp["robustness_score"] - baseline_best["robustness_score"]
        
        comparison_data.append(
            {
                "Feature Set": feature_set_name,
                "Baseline Model": baseline_best["model"],
                "Baseline F1": f"{baseline_best['macro_f1_mean']:.3f}",
                "MLP F1": f"{mlp['macro_f1_mean']:.3f}",
                "ΔF1": f"{delta_f1:+.3f}",
                "Baseline Robust": f"{baseline_best['robustness_score']:.3f}",
                "MLP Robust": f"{mlp['robustness_score']:.3f}",
                "ΔRobust": f"{delta_robust:+.3f}",
            }
        )
    
    comparison_table = pd.DataFrame(comparison_data)
    print("\n" + "=" * 100)
    print("COMPARISON: MLP vs. Baseline Models (Dataset A)")
    print("=" * 100)
    print(comparison_table.to_string(index=False))
else:
    warnings.warn(f"Baseline summary not found at {baseline_path}. Skipping comparison.")

# ============================================================================
# OPTIONAL: LOAD DATASET B COMPARISON (if available)
# ============================================================================
dataset_b_path = RESULTS_DIR / "dataset_B_3class_metrics_summary.csv"
if dataset_b_path.exists():
    print("\n" + "=" * 100)
    print("⚠️  CAVEAT: Dataset B Comparison")
    print("=" * 100)
    print("Dataset B uses 3 classes (subset of Dataset A) and different speed regimes.")
    print("Results are NOT directly comparable; shown for reference only.")
    print("-" * 100)
    
    dataset_b_df = pd.read_csv(dataset_b_path)
    print("\nDataset B Results (top model, all features):")
    db_all = dataset_b_df[dataset_b_df["feature_set"] == "all"]
    if not db_all.empty:
        print(db_all[[
            "model", "macro_f1_mean", "macro_f1_std", "worst_fold_macro_f1", "robustness_score"
        ]].head(1).to_string(index=False))

# ============================================================================
# FINAL THESIS SUMMARY
# ============================================================================
print("\n" + "=" * 100)
print("THESIS SUMMARY: Multi-Layer Perceptron for Terrain Classification")
print("=" * 100)

best_row = summary_ranked.iloc[0]

print(f"""
EXPERIMENTAL SETUP:
  Dataset: Dataset A (5 terrain classes)
  Split Strategy: Leave-One-Run-Out Cross-Validation (LORO-CV)
  Folds: {len(run_ids)} runs (run 3 split temporally into partA + partB)
  Total Windows: {len(df):,}
  
  Feature Sets Evaluated: {len(feature_sets)}
    - all: {len(feature_cols_all)} features
    - top_mi: {N_TOP_MI} top-MI features (fold-local computation)
{f"    - all_no_speed: {len(feature_sets.get('all_no_speed', []))} features (speed ablation)" if 'all_no_speed' in feature_sets else ""}

MODEL ARCHITECTURE:
  Type: Multi-Layer Perceptron (PyTorch)
  Hidden layers: {HIDDEN_DIMS}
  Batch normalization: {USE_BATCH_NORM}
  Dropout rate: {DROPOUT_RATE}
  
TRAINING CONFIGURATION:
  Optimizer: Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})
  Batch size: {BATCH_SIZE}
  Max epochs: {MAX_EPOCHS}
  Early stopping patience: {EARLY_STOPPING_PATIENCE}
  Validation split: {VAL_SPLIT * 100:.1f}% (stratified, inside training fold)
  
  ✓ Validation split carved from training fold ONLY
  ✓ Never overlaps with held-out test run
  ✓ StandardScaler fitted on training data only (no leakage)
  ✓ LabelEncoder fitted on training labels only (no leakage)

BEST RESULT:
  Feature set: {best_row["feature_set"]}
  Macro F1 (mean): {best_row["macro_f1_mean"]:.3f} ± {best_row["macro_f1_std"]:.3f}
  Macro F1 (median): {best_row["macro_f1_median"]:.3f}
  Robustness score: {best_row["robustness_score"]:.3f}
  Worst fold macro F1: {best_row["worst_fold_macro_f1"]:.3f}
  Evaluable folds: {int(best_row['evaluable_folds'])}/{len(run_ids)}

OUTPUTS SAVED:
  Results:
    - {RESULTS_DIR / "mlp_metrics_per_fold.csv"}
    - {RESULTS_DIR / "mlp_metrics_summary.csv"}
    - {RESULTS_DIR / "mlp_metrics_ranked_by_robustness.csv"}
    - {RESULTS_DIR / "mlp_per_class_metrics.csv"}
  
  Visualizations:
    - {REPORTS_DIR / "mlp_learning_curves_*.png"} (learning curves per feature set)
    - {REPORTS_DIR / "mlp_confusion_matrix_*_per_fold.png"} (per-fold matrices)
    - {REPORTS_DIR / "mlp_confusion_matrix_*_aggregated.png"} (aggregated matrices)
    - {REPORTS_DIR / "mlp_feature_importance_permutation.png"} (feature importance)
""")

print("=" * 100)
print("Notebook complete.")
print("=" * 100)



COMPARISON: MLP vs. Baseline Models (Dataset A)
 Feature Set     Baseline Model Baseline F1 MLP F1    ΔF1 Baseline Robust MLP Robust ΔRobust
         all            XGBoost       0.768  0.756 -0.012           0.704      0.700  -0.004
      top_mi LogisticRegression       0.775  0.766 -0.009           0.714      0.706  -0.008
all_no_speed            XGBoost       0.765  0.754 -0.011           0.700      0.697  -0.003

⚠️  CAVEAT: Dataset B Comparison
Dataset B uses 3 classes (subset of Dataset A) and different speed regimes.
Results are NOT directly comparable; shown for reference only.
----------------------------------------------------------------------------------------------------

Dataset B Results (top model, all features):
             model  macro_f1_mean  macro_f1_std  worst_fold_macro_f1  robustness_score
LogisticRegression       0.970239      0.020614             0.939835          0.959932

THESIS SUMMARY: Multi-Layer Perceptron for Terrain Classification

EXPERIMENTAL SETU

## Cell 10: Hyperparameter Tuning

Grid search over MLP architecture and training hyperparameters using full LORO-CV.
Each candidate configuration is evaluated with the same leak-free pipeline.
The best configuration is selected by `robustness_score = macro_f1_mean - 0.5 * macro_f1_std`.

In [11]:
# ============================================================================
# HYPERPARAMETER TUNING — full LORO-CV grid search
# ============================================================================
# Strategy: define a compact candidate grid, run full LORO-CV for each config
# on the "all" feature set, then rank by robustness_score.
# All preprocessing (scaling, label encoding, class weights) is re-fitted
# inside each fold to prevent any data leakage.
# ============================================================================

HP_SEARCH_FEATURE_SET = "all"   # tune on the full feature set only

HP_GRID = [
    # (label, hidden_dims, dropout_rate, learning_rate, weight_decay)
    # Baseline config (already run above — included for reference)
    {"label": "256-128-64 | drop=0.3 | lr=1e-3 | wd=1e-4", "hidden_dims": [256, 128, 64], "dropout_rate": 0.3, "learning_rate": 1e-3, "weight_decay": 1e-4},
    # Wider network
    {"label": "512-256-128 | drop=0.3 | lr=1e-3 | wd=1e-4", "hidden_dims": [512, 256, 128], "dropout_rate": 0.3, "learning_rate": 1e-3, "weight_decay": 1e-4},
    # Shallower / smaller network
    {"label": "128-64     | drop=0.3 | lr=1e-3 | wd=1e-4", "hidden_dims": [128, 64],       "dropout_rate": 0.3, "learning_rate": 1e-3, "weight_decay": 1e-4},
    # Higher dropout
    {"label": "256-128-64 | drop=0.5 | lr=1e-3 | wd=1e-4", "hidden_dims": [256, 128, 64], "dropout_rate": 0.5, "learning_rate": 1e-3, "weight_decay": 1e-4},
    # Lower LR
    {"label": "256-128-64 | drop=0.3 | lr=5e-4 | wd=1e-4", "hidden_dims": [256, 128, 64], "dropout_rate": 0.3, "learning_rate": 5e-4, "weight_decay": 1e-4},
    # Stronger L2
    {"label": "256-128-64 | drop=0.3 | lr=1e-3 | wd=1e-3", "hidden_dims": [256, 128, 64], "dropout_rate": 0.3, "learning_rate": 1e-3, "weight_decay": 1e-3},
]

print("=" * 80)
print(f"HYPERPARAMETER TUNING — {len(HP_GRID)} configurations × {len(run_ids)} folds")
print(f"Feature set: {HP_SEARCH_FEATURE_SET}")
print("=" * 80)

hp_base_cols = feature_sets[HP_SEARCH_FEATURE_SET]
hp_detail_rows = []

for cfg in HP_GRID:
    cfg_label     = cfg["label"]
    hidden_dims_  = cfg["hidden_dims"]
    dropout_rate_ = cfg["dropout_rate"]
    lr_           = cfg["learning_rate"]
    wd_           = cfg["weight_decay"]

    print(f"\n  Config: {cfg_label}")

    for test_run in run_ids:
        train_mask = df[RUN_COL].astype(str) != test_run
        test_mask  = ~train_mask

        y_train_hp = df.loc[train_mask, LABEL_COL].astype(str).to_numpy()
        y_test_hp  = df.loc[test_mask,  LABEL_COL].astype(str).to_numpy()

        # Feature selection (same MI logic as main loop)
        sel_cols = list(hp_base_cols)
        if HP_SEARCH_FEATURE_SET == "top_mi" and ENABLE_FOLD_LOCAL_TOP_MI:
            sel_cols = fold_local_top_mi_features(
                train_df=df.loc[train_mask],
                y_train=y_train_hp,
                pool_cols=feature_cols_all,
                n_top=N_TOP_MI,
                n_neighbors=MI_NEIGHBORS,
                random_state=RANDOM_STATE,
                fallback_cols=feature_cols_mi,
            )

        X_tr_raw = df.loc[train_mask, sel_cols]
        X_te_raw = df.loc[test_mask,  sel_cols]

        scaler_ = StandardScaler()
        X_tr_s  = scaler_.fit_transform(X_tr_raw)
        X_te_s  = scaler_.transform(X_te_raw)

        X_tr, X_vl, y_tr, y_vl = train_test_split(
            X_tr_s, y_train_hp,
            test_size=VAL_SPLIT, stratify=y_train_hp, random_state=RANDOM_STATE,
        )

        le_ = LabelEncoder()
        y_tr_enc = le_.fit_transform(y_tr)
        y_vl_enc = le_.transform(y_vl)
        y_te_enc = le_.transform(y_test_hp)

        # Class weights
        nc_  = len(le_.classes_)
        cc_  = np.bincount(y_tr_enc, minlength=nc_).astype(float)
        cw_  = torch.tensor(len(y_tr_enc) / (nc_ * np.maximum(cc_, 1)), dtype=torch.float32)

        m_ = TerrainMLP(
            input_dim=X_tr.shape[1],
            hidden_dims=hidden_dims_,
            num_classes=nc_,
            dropout_rate=dropout_rate_,
            use_batch_norm=USE_BATCH_NORM,
        ).to(device)

        cfg_ = {
            "learning_rate":           lr_,
            "batch_size":              BATCH_SIZE,
            "max_epochs":              MAX_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "weight_decay":            wd_,
            "use_lr_scheduler":        USE_LR_SCHEDULER,
            "use_class_weights":       USE_CLASS_WEIGHTS,
            "class_weights":           cw_,
        }

        trained_m_, hist_ = train_fold(
            model=m_, X_train=X_tr, y_train=y_tr_enc,
            X_val=X_vl, y_val=y_vl_enc,
            config=cfg_, device=device,
        )

        crit_ = nn.CrossEntropyLoss()
        te_ds  = TerrainDataset(X_te_s, y_te_enc)
        te_ld  = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False)
        _, y_pred_enc_ = evaluate(trained_m_, te_ld, crit_, device)
        y_pred_ = le_.inverse_transform(y_pred_enc_)

        eval_cls_ = sorted(set(y_train_hp).intersection(set(y_test_hp)))
        fm_ = compute_fold_metrics(
            y_true=y_test_hp, y_pred=y_pred_,
            evaluable_classes=eval_cls_, label_order=LABEL_ORDER,
        )

        print(
            f"    Fold {test_run} | acc={fm_['accuracy']:.3f} | macro_f1={fm_['macro_f1']:.3f}"
            f" | best_epoch={hist_['best_epoch']}"
        )

        hp_detail_rows.append({
            "config":      cfg_label,
            "hidden_dims": str(hidden_dims_),
            "dropout":     dropout_rate_,
            "lr":          lr_,
            "weight_decay": wd_,
            "test_run":    test_run,
            "accuracy":    fm_["accuracy"],
            "macro_f1":    fm_["macro_f1"],
            "best_epoch":  hist_["best_epoch"],
        })

# ── Aggregate tuning results ─────────────────────────────────────────────────
hp_detail_df = pd.DataFrame(hp_detail_rows)

def _hp_summarize(group):
    mf1 = group["macro_f1"].dropna()
    acc  = group["accuracy"].dropna()
    mean_ = float(mf1.mean()) if not mf1.empty else float("nan")
    std_  = float(mf1.std(ddof=0)) if not mf1.empty else float("nan")
    rob_  = mean_ - 0.5 * std_ if not (pd.isna(mean_) or pd.isna(std_)) else float("nan")
    return pd.Series({
        "macro_f1_mean":      mean_,
        "macro_f1_std":       std_,
        "macro_f1_median":    float(mf1.median()) if not mf1.empty else float("nan"),
        "worst_fold_macro_f1": float(mf1.min())   if not mf1.empty else float("nan"),
        "robustness_score":   rob_,
        "accuracy_mean":      float(acc.mean())   if not acc.empty else float("nan"),
        "evaluable_folds":    int(mf1.shape[0]),
    })

hp_summary_df = (
    hp_detail_df
    .groupby(["config", "hidden_dims", "dropout", "lr", "weight_decay"], as_index=False)
    .apply(_hp_summarize)
    .reset_index(drop=True)
    .sort_values("robustness_score", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("HYPERPARAMETER TUNING RESULTS (ranked by robustness_score)")
print("=" * 100)
print(hp_summary_df[[
    "config", "macro_f1_mean", "macro_f1_std", "worst_fold_macro_f1", "robustness_score", "accuracy_mean"
]].to_string(index=False))

# ── Save tuning results ───────────────────────────────────────────────────────
hp_detail_out  = RESULTS_DIR / "mlp_hp_tuning_per_fold.csv"
hp_summary_out = RESULTS_DIR / "mlp_hp_tuning_summary.csv"
hp_detail_df.to_csv(hp_detail_out,  index=False)
hp_summary_df.to_csv(hp_summary_out, index=False)
print(f"\nSaved: {hp_detail_out}")
print(f"Saved: {hp_summary_out}")

# ── Best config ───────────────────────────────────────────────────────────────
best_hp_row = hp_summary_df.iloc[0]
best_hp_config_label = str(best_hp_row["config"])
print(f"\nBest config: {best_hp_config_label}")
print(f"  macro_f1_mean={best_hp_row['macro_f1_mean']:.3f}  "
      f"macro_f1_std={best_hp_row['macro_f1_std']:.3f}  "
      f"robustness={best_hp_row['robustness_score']:.3f}  "
      f"worst_fold={best_hp_row['worst_fold_macro_f1']:.3f}")

# ── Bar chart: robustness_score per config ────────────────────────────────────
fig_hp, ax_hp = plt.subplots(figsize=(10, 5), constrained_layout=True)
colors_ = ["gold" if r == best_hp_config_label else "steelblue"
           for r in hp_summary_df["config"]]
ax_hp.barh(hp_summary_df["config"], hp_summary_df["robustness_score"],
           color=colors_, alpha=0.85)
ax_hp.axvline(hp_summary_df["robustness_score"].max(), color="gold",
              linestyle="--", linewidth=1.2, alpha=0.7)
ax_hp.set_xlabel("Robustness score  (macro_f1_mean − 0.5 × macro_f1_std)")
ax_hp.set_title("MLP Hyperparameter Tuning — Robustness by Config (LORO-CV)")
ax_hp.grid(True, axis="x", alpha=0.3)
hp_plot_path = REPORTS_DIR / "mlp_hp_tuning_robustness.png"
fig_hp.savefig(hp_plot_path, dpi=150, bbox_inches="tight")
plt.close(fig_hp)
print(f"Saved: {hp_plot_path}")


HYPERPARAMETER TUNING — 6 configurations × 5 folds
Feature set: all

  Config: 256-128-64 | drop=0.3 | lr=1e-3 | wd=1e-4
    Fold log_20260223_142511.490 | acc=0.689 | macro_f1=0.623 | best_epoch=73
    Fold log_20260226_102148.990 | acc=0.848 | macro_f1=0.843 | best_epoch=47
    Fold log_20260309_141435.414_partA | acc=0.860 | macro_f1=0.890 | best_epoch=45
    Fold log_20260309_141435.414_partB | acc=0.877 | macro_f1=0.831 | best_epoch=19
    Fold log_20260326_120021.508 | acc=0.455 | macro_f1=0.650 | best_epoch=44

  Config: 512-256-128 | drop=0.3 | lr=1e-3 | wd=1e-4
    Fold log_20260223_142511.490 | acc=0.689 | macro_f1=0.604 | best_epoch=33
    Fold log_20260226_102148.990 | acc=0.807 | macro_f1=0.796 | best_epoch=27
    Fold log_20260309_141435.414_partA | acc=0.843 | macro_f1=0.873 | best_epoch=36
    Fold log_20260309_141435.414_partB | acc=0.892 | macro_f1=0.848 | best_epoch=20
    Fold log_20260326_120021.508 | acc=0.466 | macro_f1=0.661 | best_epoch=31

  Config: 128-64    

## Cell 11: Cross-Model Comparison

Compare the best MLP configuration against all baseline and tuned baseline models
from `07_baselines.ipynb` and `08_base_tuning.ipynb`.

In [12]:
# ============================================================================
# CROSS-MODEL COMPARISON: MLP (baseline) vs MLP (best HP) vs 07 vs 08
# ============================================================================

baseline_summary_path = RESULTS_DIR / "baseline_metrics_summary.csv"
tuned_summary_path    = RESULTS_DIR / "tuned_metrics_summary.csv"

comparison_rows = []

# ── Helper to add a model family's best row for each feature set ─────────────
def _add_best_rows(df_src, model_tag, feature_sets_to_check=("all", "top_mi")):
    if df_src is None or df_src.empty:
        return
    for fs in feature_sets_to_check:
        sub = df_src[df_src["feature_set"] == fs]
        if sub.empty:
            continue
        # pick the row with highest robustness_score (break ties by macro_f1_mean)
        best = sub.sort_values(
            ["robustness_score", "macro_f1_mean"], ascending=False
        ).iloc[0]
        comparison_rows.append({
            "Model":             model_tag + f"  [{best.get('model', '')}]".rstrip(" []"),
            "Feature set":       fs,
            "Macro F1 mean":     best["macro_f1_mean"],
            "Macro F1 std":      best["macro_f1_std"],
            "Worst fold F1":     best["worst_fold_macro_f1"],
            "Robustness score":  best["robustness_score"],
            "Accuracy mean":     best["accuracy_mean"],
        })

# ── Load baseline results from 07 ────────────────────────────────────────────
baseline_df_07 = None
if baseline_summary_path.exists():
    baseline_df_07 = pd.read_csv(baseline_summary_path)
    _add_best_rows(baseline_df_07, "07 Baseline")
else:
    warnings.warn(f"baseline_metrics_summary.csv not found at {baseline_summary_path}")

# ── Load tuned results from 08 ───────────────────────────────────────────────
tuned_df_08 = None
if tuned_summary_path.exists():
    tuned_df_08 = pd.read_csv(tuned_summary_path)
    _add_best_rows(tuned_df_08, "08 Tuned")
else:
    warnings.warn(f"tuned_metrics_summary.csv not found at {tuned_summary_path}")

# ── MLP baseline (from this notebook, Cell 5) ────────────────────────────────
for fs in summary_df["feature_set"].unique():
    sub = summary_df[summary_df["feature_set"] == fs]
    if sub.empty:
        continue
    r = sub.iloc[0]
    comparison_rows.append({
        "Model":            "09 MLP baseline",
        "Feature set":      fs,
        "Macro F1 mean":    r["macro_f1_mean"],
        "Macro F1 std":     r["macro_f1_std"],
        "Worst fold F1":    r["worst_fold_macro_f1"],
        "Robustness score": r["robustness_score"],
        "Accuracy mean":    r["accuracy_mean"],
    })

# ── MLP best HP-tuned config (from Cell 10) ──────────────────────────────────
# Re-use hp_summary_df computed above (best config on "all" features)
if not hp_summary_df.empty:
    hp_best = hp_summary_df.iloc[0]
    comparison_rows.append({
        "Model":            "09 MLP HP-tuned",
        "Feature set":      HP_SEARCH_FEATURE_SET,
        "Macro F1 mean":    hp_best["macro_f1_mean"],
        "Macro F1 std":     hp_best["macro_f1_std"],
        "Worst fold F1":    hp_best["worst_fold_macro_f1"],
        "Robustness score": hp_best["robustness_score"],
        "Accuracy mean":    hp_best["accuracy_mean"],
    })

# ── Build and display comparison table ───────────────────────────────────────
comparison_table = pd.DataFrame(comparison_rows)

# Format for display
def _fmt(v):
    return f"{v:.3f}" if pd.notna(v) else "n/a"

disp = comparison_table.copy()
for col in ["Macro F1 mean", "Macro F1 std", "Worst fold F1", "Robustness score", "Accuracy mean"]:
    disp[col] = disp[col].map(_fmt)

print("\n" + "=" * 110)
print("CROSS-MODEL COMPARISON: MLP vs Baselines (07) vs Tuned Baselines (08)")
print("Ranked by Robustness score = macro_f1_mean − 0.5 × macro_f1_std")
print("=" * 110)
# Sort by robustness score numerically before displaying
comparison_table_sorted = comparison_table.sort_values("Robustness score", ascending=False).reset_index(drop=True)
disp_sorted = comparison_table_sorted.copy()
for col in ["Macro F1 mean", "Macro F1 std", "Worst fold F1", "Robustness score", "Accuracy mean"]:
    disp_sorted[col] = disp_sorted[col].map(_fmt)
print(disp_sorted.to_string(index=False))

# ── Save comparison table ─────────────────────────────────────────────────────
comp_out = RESULTS_DIR / "mlp_vs_all_baselines_comparison.csv"
comparison_table_sorted.to_csv(comp_out, index=False)
print(f"\nSaved: {comp_out}")

# ── Bar chart: macro_f1_mean + std error bars ─────────────────────────────────
fig_cmp, ax_cmp = plt.subplots(figsize=(12, max(4, len(comparison_table_sorted) * 0.6)), constrained_layout=True)

colors_cmp = []
for m in comparison_table_sorted["Model"]:
    if "MLP HP-tuned" in m:
        colors_cmp.append("gold")
    elif "MLP" in m:
        colors_cmp.append("steelblue")
    elif "08 Tuned" in m:
        colors_cmp.append("mediumseagreen")
    else:
        colors_cmp.append("lightcoral")

labels_cmp = [
    f"{r['Model']} ({r['Feature set']})"
    for _, r in comparison_table_sorted.iterrows()
]
ax_cmp.barh(
    labels_cmp,
    comparison_table_sorted["Macro F1 mean"],
    xerr=comparison_table_sorted["Macro F1 std"],
    color=colors_cmp,
    alpha=0.85,
    capsize=4,
)
ax_cmp.set_xlabel("Macro F1 mean ± std  (LORO-CV)")
ax_cmp.set_title("Cross-Model Comparison: Macro F1 by Model and Feature Set")
ax_cmp.grid(True, axis="x", alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="gold",         label="MLP HP-tuned (best)"),
    Patch(facecolor="steelblue",    label="MLP baseline"),
    Patch(facecolor="mediumseagreen", label="08 Tuned baselines"),
    Patch(facecolor="lightcoral",   label="07 Baselines"),
]
ax_cmp.legend(handles=legend_elements, loc="lower right", fontsize=9)

cmp_plot_path = REPORTS_DIR / "mlp_vs_all_baselines_comparison.png"
fig_cmp.savefig(cmp_plot_path, dpi=150, bbox_inches="tight")
plt.close(fig_cmp)
print(f"Saved: {cmp_plot_path}")

# ── Delta table: MLP HP-tuned vs best baseline from 07 ───────────────────────
if baseline_df_07 is not None:
    print("\n" + "=" * 80)
    print("DELTA: MLP HP-tuned vs best 07-baseline (per feature set)")
    print("=" * 80)
    for fs in [HP_SEARCH_FEATURE_SET]:
        mlp_rows = comparison_table_sorted[
            (comparison_table_sorted["Model"] == "09 MLP HP-tuned") &
            (comparison_table_sorted["Feature set"] == fs)
        ]
        base_rows = comparison_table_sorted[
            (comparison_table_sorted["Model"].str.startswith("07")) &
            (comparison_table_sorted["Feature set"] == fs)
        ]
        if mlp_rows.empty or base_rows.empty:
            continue
        mlp_f1   = float(mlp_rows.iloc[0]["Macro F1 mean"])
        base_f1  = float(base_rows.sort_values("Macro F1 mean", ascending=False).iloc[0]["Macro F1 mean"])
        base_mdl = base_rows.sort_values("Macro F1 mean", ascending=False).iloc[0]["Model"]
        delta    = mlp_f1 - base_f1
        print(f"  Feature set={fs}: MLP HP-tuned={mlp_f1:.3f}  best-baseline={base_f1:.3f} ({base_mdl})  Δ={delta:+.3f}")



CROSS-MODEL COMPARISON: MLP vs Baselines (07) vs Tuned Baselines (08)
Ranked by Robustness score = macro_f1_mean − 0.5 × macro_f1_std
                           Model  Feature set Macro F1 mean Macro F1 std Worst fold F1 Robustness score Accuracy mean
                 09 MLP HP-tuned          all         0.782        0.125         0.614            0.720         0.758
07 Baseline  [LogisticRegression       top_mi         0.775        0.122         0.626            0.714         0.742
   08 Tuned  [LogisticRegression          all         0.765        0.109         0.626            0.711         0.741
                 09 MLP baseline       top_mi         0.766        0.119         0.599            0.706         0.756
           07 Baseline  [XGBoost          all         0.768        0.128         0.578            0.704         0.739
              08 Tuned  [XGBoost       top_mi         0.765        0.128         0.579            0.701         0.723
                 09 MLP baseline       